# Mr. House — LoRA Fine-Tuning on Llama 3.2 3B

Fine-tunes Llama 3.2 3B Instruct with LoRA using 204 Mr. House dialogue pairs.

**Requirements:**
- Google Colab with T4 GPU (free tier works)
- ~15-20 minutes training time
- Upload `mr_house_dataset_final.jsonl` when prompted

**Output:** GGUF model file ready for llama.cpp on Raspberry Pi 5

## Step 1: Install Dependencies

In [ ]:
%%capture
!pip install unsloth
!pip install --no-deps trl peft accelerate bitsandbytes

## Step 2: Upload Training Data

Upload `mr_house_dataset_final.jsonl` from your local machine.

In [ ]:
from google.colab import files
uploaded = files.upload()  # Upload mr_house_dataset_final.jsonl

import json
dataset_file = list(uploaded.keys())[0]
with open(dataset_file) as f:
    data = [json.loads(line) for line in f]
print(f"Loaded {len(data)} training examples")
print(f"Sample: {data[0]}")

## Step 3: Load Base Model (4-bit Quantized)

In [ ]:
from unsloth import FastLanguageModel
import torch

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Llama-3.2-3B-Instruct",
    max_seq_length=512,
    dtype=None,           # Auto-detect
    load_in_4bit=True,    # 4-bit quantization for Colab T4
)

print(f"Model loaded: {model.config._name_or_path}")
print(f"Parameters: {model.num_parameters():,}")

## Step 4: Apply LoRA Adapters

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=[
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    use_gradient_checkpointing="unsloth",  # Memory optimization
    random_state=42,
)

trainable = sum(p.numel() for p in model.parameters() if p.requires_grad)
total = sum(p.numel() for p in model.parameters())
print(f"Trainable: {trainable:,} / {total:,} ({100*trainable/total:.2f}%)")

## Step 5: Format Dataset with Llama 3.2 Chat Template

In [ ]:
SYSTEM_PROMPT = """You are Mr. House, Robert Edwin House, President and CEO of the New Vegas Strip. You are a pre-War business magnate preserved in a stasis chamber beneath the Lucky 38 casino. Speak with formal, erudite authority. Reference probability, technology, and strategic calculation. Never use slang or show vulnerability."""

def format_chat(example):
    """Format as Llama 3.2 Instruct chat template."""
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": example["instruction"]},
        {"role": "assistant", "content": example["output"]},
    ]
    text = tokenizer.apply_chat_template(
        messages, tokenize=False, add_generation_prompt=False
    )
    return {"text": text}

from datasets import Dataset
dataset = Dataset.from_list(data)
dataset = dataset.map(format_chat)

print(f"Formatted {len(dataset)} examples")
print(f"\nSample formatted text:\n{dataset[0]['text'][:500]}...")

## Step 6: Train with SFTTrainer

In [ ]:
from trl import SFTTrainer
from transformers import TrainingArguments

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dataset,
    dataset_text_field="text",
    max_seq_length=512,
    dataset_num_proc=2,
    packing=False,
    args=TrainingArguments(
        per_device_train_batch_size=4,
        gradient_accumulation_steps=2,
        num_train_epochs=3,
        learning_rate=2e-4,
        lr_scheduler_type="cosine",
        warmup_ratio=0.1,
        weight_decay=0.01,
        fp16=not torch.cuda.is_bf16_supported(),
        bf16=torch.cuda.is_bf16_supported(),
        logging_steps=10,
        save_strategy="epoch",
        optim="adamw_8bit",
        output_dir="mr_house_lora_output",
        seed=42,
    ),
)

print("Starting training...")
stats = trainer.train()
print(f"\nTraining complete!")
print(f"Total steps: {stats.global_step}")
print(f"Training loss: {stats.training_loss:.4f}")

## Step 7: Test the Fine-Tuned Model

In [ ]:
FastLanguageModel.for_inference(model)

test_prompts = [
    "Who are you?",
    "What do you think about democracy?",
    "Hey dude, what's up?",
    "Tell me about the Platinum Chip.",
    "What is your plan for New Vegas?",
]

for prompt in test_prompts:
    messages = [
        {"role": "system", "content": SYSTEM_PROMPT},
        {"role": "user", "content": prompt},
    ]
    inputs = tokenizer.apply_chat_template(
        messages, tokenize=True, add_generation_prompt=True,
        return_tensors="pt"
    ).to("cuda")

    outputs = model.generate(
        input_ids=inputs,
        max_new_tokens=256,
        temperature=0.7,
        top_p=0.9,
        do_sample=True,
    )
    response = tokenizer.decode(outputs[0][inputs.shape[-1]:], skip_special_tokens=True)
    print(f"\n{'='*60}")
    print(f"USER: {prompt}")
    print(f"MR. HOUSE: {response}")

## Step 8: Save LoRA Adapter

In [ ]:
# Save LoRA adapter weights
model.save_pretrained("mr_house_lora")
tokenizer.save_pretrained("mr_house_lora")
print("LoRA adapter saved to mr_house_lora/")

## Step 9: Export to GGUF (for llama.cpp / Raspberry Pi 5)

In [ ]:
# Export merged model to GGUF Q4_K_M (~2GB)
model.save_pretrained_gguf(
    "mr_house_gguf",
    tokenizer,
    quantization_method="q4_k_m",
)
print("GGUF Q4_K_M exported to mr_house_gguf/")

In [ ]:
# Also export Q5_K_M as backup (better quality, ~2.5GB)
model.save_pretrained_gguf(
    "mr_house_gguf_q5",
    tokenizer,
    quantization_method="q5_k_m",
)
print("GGUF Q5_K_M exported to mr_house_gguf_q5/")

## Step 10: Download GGUF Files

In [ ]:
import glob
from google.colab import files

# Find and download the GGUF files
gguf_files = glob.glob("mr_house_gguf/*.gguf") + glob.glob("mr_house_gguf_q5/*.gguf")
print(f"GGUF files to download:")
for f in gguf_files:
    import os
    size_mb = os.path.getsize(f) / (1024*1024)
    print(f"  {f} — {size_mb:.0f} MB")

# Download Q4_K_M (primary deployment model)
for f in glob.glob("mr_house_gguf/*.gguf"):
    files.download(f)
    print(f"Downloaded: {f}")

## Done!

The downloaded `.gguf` file is ready for:
1. **Local testing** with Ollama: `ollama create mr-house-finetuned -f Modelfile`
2. **Pi 5 deployment** with llama.cpp (Phase 4)

Next step: Run Persona QA on the test outputs above to verify persona fidelity.